# 48-bit MLP memorizer dataset — 16 strings, realistic (iid) sampling

Generates a dataset of tiny MLPs that each memorize **16** random 48-bit strings
(binary membership classifier: `logit(x) = b2 + W2·relu(W1·x + b1)`).

**Changes vs the original generator** (which got only ~17% clean memorization):
- `N_TARGETS`: 32 → **16** (lighter memorization load per hidden unit)
- data is **iid samples from an implicit mixture distribution** — each example is a
  memorized string w.p. `POS_FRAC` (else a uniform random negative), so the number of
  positives and the per-string counts **vary every minibatch** (Binomial/Multinomial),
  instead of a rigid 256-positive / 256-negative batch.
- kept: `H=128`, `EPOCHS=6000`, pure Adam, no weight decay.

Under the GCG / simulated-annealing global-maximum test this reaches **~70%** clean
memorizers (vs ~17% for the original 32-string / H=128 setup). Note: **more steps hurts**
(cross-entropy logit blow-up in the unsampled space) — 6000 is the sweet spot.

**Kaggle setup:** Notebook → Settings → Accelerator = **GPU T4 x2**.
Output chunks are written to `/kaggle/working/dataset/` and saved with the notebook.
Each chunk holds `B_PER_GPU=1024` MLPs; the 16 memorized strings are `templates[:, :, 1:]`.

In [1]:
%%writefile dataset_generator_16.py
import os
import time
import math
import torch
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp

# ---------------------------------------------------------
# HYPERPARAMETERS
#   Changes vs the original 48-bit generator:
#     * N_TARGETS 32 -> 16   (lighter memorization load)
#     * data is now iid samples from an implicit mixture distribution
#       (per-minibatch positive / per-target counts VARY), instead of a
#       rigid 256 positive / 256 negative split.
#   Kept: D=48, H=128, EPOCHS=6000, lr 5e-3 -> 1e-4, pure Adam (no weight decay).
#   This setup memorizes "cleanly" (32 strings are the global score maxima)
#   for ~70% of MLPs under the GCG/SA verification test.
# ---------------------------------------------------------
B_PER_GPU    = 1024      # independent MLPs trained in parallel per GPU per chunk
N_TARGETS    = 16        # memorized strings per MLP
D            = 48        # input dimension (bits)
H            = 128       # hidden width
BATCH_SIZE   = 512       # examples per optimization step (iid from the mixture)
POS_FRAC     = 0.5       # P(an example is a memorized string)
EPOCHS       = 6000      # steps per MLP  (more -> cross-entropy logit blow-up -> FEWER pass)
LR_START     = 0.005
LR_END       = 0.0001
RUN_DURATION = 3600      # wall-clock seconds of generation per GPU process
OUT_DIR      = "/kaggle/working/dataset"


def gpu_worker(rank, world_size, run_duration):
    device = torch.device(f"cuda:{rank}")
    torch.cuda.set_device(device)
    print(f"[GPU {rank}] worker initialized.", flush=True)
    os.makedirs(OUT_DIR, exist_ok=True)

    # int64 shifts to unpack uniform 48-bit negatives; per-MLP row index for gather
    shifts    = torch.arange(D, dtype=torch.int64, device=device)
    batch_idx = torch.arange(B_PER_GPU, device=device).unsqueeze(1)

    start_run = time.time()
    chunk_idx = 0
    while (time.time() - start_run) < run_duration:
        chunk_start = time.time()

        # ---- fresh batch of independent MLPs ----
        W1 = torch.randn(B_PER_GPU, H, D, device=device) / math.sqrt(D)
        b1 = torch.zeros(B_PER_GPU, H, device=device)
        W2 = torch.randn(B_PER_GPU, 1, H, device=device) / math.sqrt(H)
        b2 = torch.zeros(B_PER_GPU, 1, device=device)
        for p in (W1, b1, W2, b2):
            p.requires_grad_(True)

        # ---- the implicit "memorized set" of each MLP ----
        targets = torch.randint(0, 2, (B_PER_GPU, N_TARGETS, D),
                                dtype=torch.float32, device=device)

        gamma = (LR_END / LR_START) ** (1.0 / EPOCHS)
        try:
            opt = optim.Adam([W1, b1, W2, b2], lr=LR_START, fused=True)
        except Exception:
            opt = optim.Adam([W1, b1, W2, b2], lr=LR_START)
        sched = optim.lr_scheduler.ExponentialLR(opt, gamma=gamma)

        for epoch in range(EPOCHS):
            # ---- iid minibatch from the implicit mixture distribution ----
            # Each (MLP, slot) is independently a POSITIVE w.p. POS_FRAC -- a uniform
            # random one of that MLP's N_TARGETS memorized strings -- otherwise a
            # uniform random 48-bit NEGATIVE. So the number of positives per MLP is
            # Binomial(BATCH_SIZE, POS_FRAC) and the per-target multiplicities are
            # Multinomial: both vary every minibatch, exactly like repeatedly drawing
            # a fresh minibatch from a big fixed dataset.
            is_pos = (torch.rand(B_PER_GPU, BATCH_SIZE, device=device) < POS_FRAC)
            choice = torch.randint(0, N_TARGETS, (B_PER_GPU, BATCH_SIZE), device=device)
            X_tgt  = targets[batch_idx, choice]                          # (B,BATCH,D)
            neg_i  = torch.randint(0, 2 ** 48, (B_PER_GPU, BATCH_SIZE),
                                   dtype=torch.int64, device=device)
            X_rand = ((neg_i.unsqueeze(-1) >> shifts) & 1).float()       # uniform 48-bit
            X = torch.where(is_pos.unsqueeze(-1), X_tgt, X_rand)         # (B,BATCH,D)
            Y = is_pos.float().unsqueeze(-1)                            # (B,BATCH,1)

            h      = F.relu(torch.bmm(X, W1.transpose(1, 2)) + b1.unsqueeze(1))
            logits = torch.bmm(h, W2.transpose(1, 2)) + b2.unsqueeze(1)

            opt.zero_grad(set_to_none=True)
            loss = F.binary_cross_entropy_with_logits(logits, Y)
            loss.backward()
            opt.step()
            sched.step()

        # ---- save chunk (drop-in compatible with verify_memorization.py) ----
        bos = torch.full((B_PER_GPU, N_TARGETS, 1), 2.0, device=device)
        templates = torch.cat([bos, targets], dim=2)                   # col0=BOS(2); [:,:,1:]=targets

        chunk = {
            "W1": W1.detach().cpu().half(),
            "b1": b1.detach().cpu().half(),
            "W2": W2.detach().cpu().half(),
            "b2": b2.detach().cpu().half(),
            "templates": templates.detach().cpu().to(torch.int8),      # (B, 16, 49)
            "config": {"D": D, "H": H, "N_TARGETS": N_TARGETS, "BATCH_SIZE": BATCH_SIZE,
                       "POS_FRAC": POS_FRAC, "EPOCHS": EPOCHS, "B_PER_GPU": B_PER_GPU},
        }
        path = f"{OUT_DIR}/chunk16_gpu{rank}_idx{chunk_idx}.pt"
        torch.save(chunk, path)
        print(f"[GPU {rank}] chunk {chunk_idx} done in {time.time()-chunk_start:.1f}s "
              f"final_loss={loss.item():.5f} -> {path}", flush=True)
        chunk_idx += 1


if __name__ == "__main__":
    mp.set_start_method("spawn", force=True)
    world_size = torch.cuda.device_count()
    print(f"Starting dataset generation on {world_size} GPU(s)...", flush=True)
    if world_size == 0:
        raise RuntimeError("No CUDA GPUs found. On Kaggle set Accelerator = 'GPU T4 x2'.")
    mp.spawn(gpu_worker, args=(world_size, RUN_DURATION), nprocs=world_size, join=True)


Writing dataset_generator_16.py


In [2]:
# Launches one process per GPU (2x T4). Runs for RUN_DURATION seconds.
!python dataset_generator_16.py

Starting dataset generation on 2 GPU(s)...
[GPU 0] worker initialized.
[GPU 1] worker initialized.
[GPU 1] chunk 0 done in 152.4s final_loss=0.00005 -> /kaggle/working/dataset/chunk16_gpu1_idx0.pt
[GPU 0] chunk 0 done in 153.5s final_loss=0.00005 -> /kaggle/working/dataset/chunk16_gpu0_idx0.pt
[GPU 1] chunk 1 done in 144.8s final_loss=0.00005 -> /kaggle/working/dataset/chunk16_gpu1_idx1.pt
[GPU 0] chunk 1 done in 146.2s final_loss=0.00005 -> /kaggle/working/dataset/chunk16_gpu0_idx1.pt
[GPU 1] chunk 2 done in 144.8s final_loss=0.00005 -> /kaggle/working/dataset/chunk16_gpu1_idx2.pt
[GPU 0] chunk 2 done in 146.2s final_loss=0.00005 -> /kaggle/working/dataset/chunk16_gpu0_idx2.pt
[GPU 1] chunk 3 done in 144.7s final_loss=0.00005 -> /kaggle/working/dataset/chunk16_gpu1_idx3.pt
[GPU 0] chunk 3 done in 146.2s final_loss=0.00005 -> /kaggle/working/dataset/chunk16_gpu0_idx3.pt
[GPU 1] chunk 4 done in 144.8s final_loss=0.00005 -> /kaggle/working/dataset/chunk16_gpu1_idx4.pt
[GPU 0] chunk 4 don

In [3]:
# ---------------------------------------------------------
# Quick GPU sanity + pass-rate estimate on the generated chunk.
# (A fast batched version of the GCG/SA global-maximum test. This is a quick
#  estimate with modest restarts -- the definitive test is verify_memorization.py.)
# ---------------------------------------------------------
import glob, torch

files = sorted(glob.glob("/kaggle/working/dataset/chunk16_*.pt"))
print(len(files), "chunk(s):", files[:3])
d = torch.load(files[0], map_location="cpu")
dev = "cuda" if torch.cuda.is_available() else "cpu"
W1 = d["W1"].to(dev).float(); b1 = d["b1"].to(dev).float()
W2 = d["W2"].to(dev).float(); b2 = d["b2"].to(dev).float().squeeze(-1)
targets = d["templates"][:, :, 1:].to(dev).float()          # (M, 16, 48)
M, T, D = targets.shape; H = W1.shape[1]
print("chunk:", M, "MLPs   H =", H, "  targets/MLP =", T)


def fwd(X, w1, bb1, w2, bb2):
    pre = torch.einsum("mrd,mhd->mrh", X, w1) + bb1[:, None, :]
    return torch.einsum("mrh,mh->mr", torch.relu(pre), w2[:, 0, :]) + bb2[:, None]


def verify(sl, restarts=512, hc_steps=60, sa_steps=400):
    w1, bb1, w2, bb2 = W1[sl], b1[sl], W2[sl], b2[sl]
    tgt = targets[sl]; m = tgt.shape[0]
    bar = fwd(tgt, w1, bb1, w2, bb2).min(1).values                       # weakest target

    # exact single-bit steepest-ascent hill-climb (random + target-seeded starts)
    X = torch.cat([(torch.rand(m, restarts, D, device=dev) < 0.5).float(), tgt.clone()], 1)
    W1col = w1.transpose(1, 2)
    for _ in range(hc_steps):
        cur = fwd(X, w1, bb1, w2, bb2)
        pre = torch.einsum("mrd,mhd->mrh", X, w1) + bb1[:, None, :]
        pre_all = pre[:, :, None, :] + (1 - 2 * X)[:, :, :, None] * W1col[:, None, :, :]
        fl = torch.einsum("mrih,mh->mri", torch.relu(pre_all), w2[:, 0, :]) + bb2[:, None, None]
        gain, bit = (fl - cur[:, :, None]).max(2)
        imp = gain > 1e-6
        sel = torch.gather(X, 2, bit[:, :, None]).squeeze(2)
        X.scatter_(2, bit[:, :, None], torch.where(imp, 1 - sel, sel)[:, :, None])
        if not imp.any():
            break

    # simulated annealing (multi-bit proposals)
    Xs = torch.cat([(torch.rand(m, restarts, D, device=dev) < 0.5).float(), tgt.clone()], 1)
    cur = fwd(Xs, w1, bb1, w2, bb2); best = cur.clone(); bestX = Xs.clone()
    for s in range(sa_steps):
        temp = 2.0 * (0.02 / 2.0) ** (s / max(1, sa_steps - 1))
        prop = (torch.rand_like(Xs) < 1.5 / D).float()
        Xp = torch.where(prop > 0, 1 - Xs, Xs)
        lp = fwd(Xp, w1, bb1, w2, bb2); dd = lp - cur
        acc = (dd > 0) | (torch.rand(m, Xs.shape[1], device=dev) < torch.exp(dd / temp))
        Xs = torch.where(acc[:, :, None], Xp, Xs); cur = torch.where(acc, lp, cur)
        bb = cur > best; best = torch.where(bb, cur, best); bestX = torch.where(bb[:, :, None], Xs, bestX)

    cand = torch.cat([X, bestX], 1)
    lo = fwd(cand, w1, bb1, w2, bb2)
    is_t = (cand[:, :, None, :] == tgt[:, None, :, :]).all(-1).any(-1)
    lo = torch.where(is_t, torch.full_like(lo, -1e9), lo)
    best_inv = lo.max(1).values
    return (best_inv < bar)                                              # per-MLP PASS bool


# also confirm positives are memorized (target probability ~ 1)
tp = torch.sigmoid(fwd(targets, W1, b1, W2, b2))
print(f"target prob: min={tp.min():.4f}  mean={tp.mean():.4f}")

n_test = min(M, 256)
passes = torch.cat([verify(slice(lo, min(lo + 32, n_test))) for lo in range(0, n_test, 32)])
print(f"PASS rate on {passes.numel()} sampled MLPs: {100 * passes.float().mean():.1f}%")


50 chunk(s): ['/kaggle/working/dataset/chunk16_gpu0_idx0.pt', '/kaggle/working/dataset/chunk16_gpu0_idx1.pt', '/kaggle/working/dataset/chunk16_gpu0_idx10.pt']
chunk: 1024 MLPs   H = 128   targets/MLP = 16
target prob: min=0.9998  mean=1.0000
PASS rate on 256 sampled MLPs: 81.6%
